In [1]:
# Import libraries
from pathlib import Path
import torch
import torch.nn as nn

from src.data.data_loader import create_dataloaders
from src.model.transformer import build_transformer
from src.model.transformer import Transformer
from src.train.training import train_model
from src.utils.utils import get_device
from nltk.tokenize import word_tokenize

from src.utils.constants import PADDING_ID, UNKNOWN_ID, START_OF_SENTENCE_ID, END_OF_SENTENCE_ID
from src.utils.constants import PADDING_VALUE, UNKNOWN_VALUE, START_OF_SENTENCE_VALUE, END_OF_SENTENCE_VALUE

/opt/anaconda3/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/opt/anaconda3/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <CFED5F8E-EC3F-36FD-AAA3-2C6C7F8D3DD9> /opt/anaconda3/lib/python3.11/site-packages/torchvision/image.so
  Expected in:     <CDAC6E34-8608-3E70-8B2F-32BCD38E90FB> /opt/anaconda3/lib/python3.11/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
# Initialize model and training parameters

# Size of embedding vector
d_model = 512
# Max sequence length for input words/tokens
seq_len = 100
# Dropout rate
dropout = 0.1
# number of encoder blocks
num_layers = 1
# number of attention heads
num_heads = 8
# Number of hidden nodes for feed-forward layer
d_ff = 4*d_model

# Number of epochs
epochs = 5
# Batch size for training
batch_size = 128

# Train file
train_file = './data/train/poems.txt'

In [3]:
# Get a device to use for training/inference
device = get_device()

# Create training and testing data loaders
train_dataloader, vocab = create_dataloaders(batch_size, seq_len, train_file)

print(f'Training data size: {len(train_dataloader) * batch_size}')

Number of tokenized words:  194655
Number of tokenized words after adding <eos>:  194755
Training data size: 194688


In [1]:
# Download model from here and save it to the models directory (Since Github doesn't store large files)
# Download link: https://drive.google.com/file/d/1U1yNw74U7RdhZAkvZQZoCJWv9yCmmjVK/view?usp=sharing

In [8]:
# Create new instance of model and load saved state dict
MODEL_PATH = Path("models")
MODEL_NAME = "07_text_generation.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

loaded_model = build_transformer(d_model, len(vocab), seq_len, dropout,
                                num_layers, num_heads, d_ff)
loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
loaded_model.to(device)

def generate_text(input, max_tokens_to_generate):
    with torch.inference_mode():
        
        output = input.clone()
        for _ in range(max_tokens_to_generate):
            
            curr_seq_len = input.size(1)
            
            if curr_seq_len > seq_len:
                input = input[:, -seq_len:]
            
            encoder_output = loaded_model.encode(input)
            y_logits = loaded_model.project(encoder_output)
            
            # for all the batches, get the embeds for last predicted sequence
            y_logits = y_logits[:, -1, :] 
            
            # for all the batches, get the embeds for last predicted sequence
            probs = y_logits.softmax(dim=1)            
            # get the probable token based on the input probs
            idx_next = torch.multinomial(probs, num_samples=1) 

            input = torch.cat([input, idx_next], dim=1)
            output = torch.cat([output, idx_next], dim=1)
            
        return output

In [10]:
text = 'Love'
input = [vocab.get(token, vocab.get(UNKNOWN_ID)) for token in word_tokenize(text)]
input_tensor = torch.tensor(input, dtype=torch.long).unsqueeze(0).to(device)

output_tensor = generate_text(input_tensor, 200).squeeze()
output_array_tokens = output_tensor.cpu().numpy()

sorted_items = sorted(vocab.items(), key=lambda item: item[1])
sorted_keys = [item[0] for item in sorted_items]

output_array_words = [sorted_keys[token] for token in output_array_tokens]
print(' '.join(output_array_words))

Love can change the world in a moment I 'll paint the picture , let me set the scene You know , the future 's on your face Hangs in the air like stars in outer space When Emma falls in love , she disappears And we all just laugh after seein ' it all these years When Emma falls apart , it 's when she 's alone She takes on the pain and bears it on her own 'Cause when Emma falls in love , she 's in it , she 's in it for keeps She wo n't walk away unless she knows she knows she absolutely has to tell me she 's the kind of book that you ca n't put down Like if Cleopatra grew up in a small town And all the bad boys would be good boys If they only had a chance to love her And to tell you the truth , sometimes I wish I was her Well , she 's so New York 'cause I met you on Grafton street right outside of the bar , She shared a cigarette with me while her brother played the guitar , She asked me


In [11]:
text = 'conquer'
input = [vocab.get(token, vocab.get(UNKNOWN_ID)) for token in word_tokenize(text)]
input_tensor = torch.tensor(input, dtype=torch.long).unsqueeze(0).to(device)

output_tensor = generate_text(input_tensor, 200).squeeze()
output_array_tokens = output_tensor.cpu().numpy()

sorted_items = sorted(vocab.items(), key=lambda item: item[1])
sorted_keys = [item[0] for item in sorted_items]

output_array_words = [sorted_keys[token] for token in output_array_tokens]
print(' '.join(output_array_words))

conquer your fear , You know hearts do n't break around here , Yeah , yeah , yeah , yeah ... You are the one , girl You know that it 's true I 'm feeling younger Every time that I 'm alone with you We were supposed to keep quiet 'cause I 'm not trying to know , If it will not give you up this now , But darling , just know You know how I said maybe We 're gon na be the one that saves me , And after all You 're my wonderwall Today was gon na be the day , But they fell in love you 'll never throw it back to fly I could never had a chance If I took a little breaks in your soul here and your fears And you know your ghosts , lost control your thought it would be us do if it And if a party Girls carrying their shoes down in the lobby Candle wax and Polaroids on the hardwood floor You and I could only find from the night before , but Do n't read the last page But I stay when you 're lost ,
